# 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


## STEP 0. 프로젝트 설정

- 라이브러리 설치 (langchain, langchain-openai, langchain-community langchain-core, langchain-text-splitters)
- 환경 설치 (dotenv)

### 필요한 라이브러리를 모두 다운받기

In [ ]:
# !pip install -U -q langchain langchain-openai
# !pip install -U -q langchain-community langchain-core
# !pip install -U langchain-text-splitters

# !pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

# !pip install dotenv

### OPENAI_API_KEY 환경설정

In [1]:
# !pip install dotenv
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")

#API key 없음 이라고 나오면 .env 파일이 현재 작업 디렉토리에 없거나, OPENAI_API_KEY 항목이 없는 것입니다. .env 파일을 확인해주세요.
print(api_key[:7] + "..." if api_key else "API key 없음")  

sk-proj...


In [ ]:
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5-nano"  # 또는 "gpt-4o-mini", "gpt-5-nano" 등 원하는 모델로 변경
TOKENIZER_NAME = "o200k_base"
EMBEDDING_MODEL_NAME = "text-embedding-3-small"  


# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.0,
)

### Step 1 Document loader

Demian 문서는 pdf파일이므로 PDFLoader를 사용합니다.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./content/Demian.pdf")
pages = loader.load_and_split()

C:\Users\Sungwoo\AppData\Local\Temp\ipykernel_51280\2482086896.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
len(pages)

182

In [6]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

### Step 2 Text splitters

#### 전처리

- 182 pages로 나오는데 실제 pdf파일의 page 수와 일치합니다.
- OCR 기반 데이터라 문장이 끝나지 않았음에도 줄바꿈이 되어있음.
- OCR 기반 데이터라 띄어쓰기가 2개이상인 곳도 있음.
- 단어도 이상하게 깨짐
  - WOR.LDS
  - ha.d
  - aJ:
- 페이지 번호가 본문과 구분없이 섞여있음.
- 다운로드 출처 문구가 반복됨.

#### 즉 Text splitters 적용하기전에 OCR 텍스트를 정리해야함. + 페이지 번호 / 출처 metadata 유지

In [7]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import tiktoken

tokenizer = tiktoken.get_encoding(TOKENIZER_NAME)

def tiktoken_len(text: str) -> int:
    return len(tokenizer.encode(text))


def clean_ocr_text(text: str) -> str:
    # 반복 출처 문구 제거
    text = re.sub(r"Downloaded from https?://\S+", "", text)

    # 페이지 번호만 단독으로 있는 줄 제거
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # 단어가 하이픈으로 줄바꿈된 경우 연결
    # 예: invent-\ned -> invented
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # 문장 중간 줄바꿈을 공백으로 변경
    # 예: I began to talk. I \ninvented -> I began to talk. I invented
    text = re.sub(r"(?<![.!?])\n(?!\n)", " ", text)

    # 남은 일반 줄바꿈 정리
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 여러 공백 정리
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


cleaned_docs = []

for page_number, page in enumerate(pages):
    cleaned_text = clean_ocr_text(page.page_content)

    if not cleaned_text:
        continue

    cleaned_docs.append(
        Document(
            page_content=cleaned_text,
            metadata={
                **page.metadata,
                "page": page_number + 1,
            }
        )
    )

In [11]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=TOKENIZER_NAME,
    chunk_size=500,
    chunk_overlap=80,
    separators=[
        "\n\n",
        ". ",
        "? ",
        "! ",
        "; ",
        ": ",
        ", ",
        " ",
        "",
    ],
)

chunks = text_splitter.split_documents(cleaned_docs)

print("pages:", len(pages))
print("cleaned_docs:", len(cleaned_docs))
print("chunks:", len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

pages: 182
cleaned_docs: 182
chunks: 182
DEMIAN •
{'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': './content/Demian.pdf', 'total_pages': 182, 'page': 1, 'page_label': '1'}


In [10]:
len(chunks)

182

In [13]:
print(chunks[0].page_content)

DEMIAN •


In [14]:
print(chunks[2].page_content)

Prologue I cannot tell my story without going a long way back. If it were possible I would go back much farther still to the very earliest years of my childhood and beyond them to my family origins. When poets write novels they are apt to behave as if they were gods, with the power to look beyond and com­ prehend any human story and serve it up as if the Almighty himself, omnipresent, were relating it in all its naked truth. That I am no more able to do than the poets. But my story is more important to me than any poet's story to him, for it is my own-and it is the story of a huffian being-not an invented, idealised person but a real, live, uniq:-e being. What constitutes a real, live human being is more of a mystery than ever these days, and men-each one of whom is a valuable, unique experiment on the part of nature-are shot down whole­ sale. If, however, we were not something more than unique human beings and each man jack of us could really be dismissed from this world with a bullet

In [17]:
length = []
for chunk in chunks:
    length.append(len(chunk.page_content))

print(length)

[8, 60, 1274, 1677, 1254, 1657, 1720, 1696, 1619, 1634, 1415, 1385, 1495, 1380, 1412, 1512, 1675, 1619, 1657, 1629, 1603, 1638, 1507, 1425, 1626, 1056, 1246, 1589, 1435, 1687, 1597, 1636, 1536, 1631, 1635, 1658, 1467, 1482, 1399, 1591, 1411, 1286, 1503, 1410, 1467, 1574, 1613, 1661, 1116, 1229, 1695, 1646, 1671, 1662, 1661, 1687, 1623, 1550, 1674, 1665, 1573, 1452, 1688, 1667, 1751, 1550, 1637, 1523, 1643, 1588, 1555, 1347, 1262, 1604, 1498, 1577, 1691, 1677, 1696, 1643, 1657, 1664, 1656, 1620, 1572, 1670, 1685, 1649, 1592, 1619, 1432, 1541, 1509, 1675, 1675, 1616, 989, 1175, 1569, 1590, 1548, 1644, 1702, 1594, 1708, 1640, 1706, 1436, 1434, 1399, 1624, 1603, 1575, 1666, 1742, 1681, 1507, 1275, 1611, 1602, 1608, 1560, 1513, 1466, 1437, 1567, 1588, 1595, 1654, 1484, 1267, 1669, 1594, 1544, 1617, 1588, 1582, 1658, 1711, 1638, 1618, 538, 1241, 1631, 1498, 1529, 1379, 1677, 1594, 1624, 1654, 1683, 1536, 1628, 1433, 1508, 1464, 1712, 1695, 1613, 1666, 1572, 1681, 1611, 1652, 1555, 1616, 1571

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding(TOKENIZER_NAME)

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [20]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk.page_content))

print(length)
print(tiktoken_length)

[8, 60, 1274, 1677, 1254, 1657, 1720, 1696, 1619, 1634, 1415, 1385, 1495, 1380, 1412, 1512, 1675, 1619, 1657, 1629, 1603, 1638, 1507, 1425, 1626, 1056, 1246, 1589, 1435, 1687, 1597, 1636, 1536, 1631, 1635, 1658, 1467, 1482, 1399, 1591, 1411, 1286, 1503, 1410, 1467, 1574, 1613, 1661, 1116, 1229, 1695, 1646, 1671, 1662, 1661, 1687, 1623, 1550, 1674, 1665, 1573, 1452, 1688, 1667, 1751, 1550, 1637, 1523, 1643, 1588, 1555, 1347, 1262, 1604, 1498, 1577, 1691, 1677, 1696, 1643, 1657, 1664, 1656, 1620, 1572, 1670, 1685, 1649, 1592, 1619, 1432, 1541, 1509, 1675, 1675, 1616, 989, 1175, 1569, 1590, 1548, 1644, 1702, 1594, 1708, 1640, 1706, 1436, 1434, 1399, 1624, 1603, 1575, 1666, 1742, 1681, 1507, 1275, 1611, 1602, 1608, 1560, 1513, 1466, 1437, 1567, 1588, 1595, 1654, 1484, 1267, 1669, 1594, 1544, 1617, 1588, 1582, 1658, 1711, 1638, 1618, 538, 1241, 1631, 1498, 1529, 1379, 1677, 1594, 1624, 1654, 1683, 1536, 1628, 1433, 1508, 1464, 1712, 1695, 1613, 1666, 1572, 1681, 1611, 1652, 1555, 1616, 1571

### Step 3 Vector Empeddings

In [21]:
import openai
client = openai.OpenAI()

In [22]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)

### 임베딩 모델 테스트

In [24]:
embeddings = embedding_model.embed_documents(
    [
        "이건 빨간 사과입니다.",
        "이건 무지개색 애플 매킨토시 입니다.",
        "이건 파란 페이스북 입니다.",
    ]
)

In [25]:
print(embeddings[1])

[-0.006839752197265625, -0.03778076171875, -0.048065185546875, 0.021575927734375, 0.0007233619689941406, -0.0621337890625, 0.03179931640625, -0.01058197021484375, -0.01064300537109375, -0.0604248046875, 0.0130462646484375, 0.010833740234375, -0.0265350341796875, 0.004833221435546875, -0.0133819580078125, -0.0282135009765625, -0.0355224609375, -0.005062103271484375, -0.01485443115234375, -0.0212860107421875, -0.0017576217651367188, -0.02191162109375, 0.005336761474609375, -3.987550735473633e-05, 0.007350921630859375, 0.00984954833984375, 0.048309326171875, -0.0672607421875, 0.0108795166015625, 0.01207733154296875, -0.04217529296875, 0.0011777877807617188, 0.0128936767578125, -0.052154541015625, -0.02410888671875, -0.040985107421875, -0.021026611328125, 0.002414703369140625, -0.0303955078125, 0.017608642578125, -0.033203125, -0.0008425712585449219, 0.005767822265625, 0.0005965232849121094, 0.048492431640625, 0.01021575927734375, 0.00447845458984375, 0.037445068359375, -0.0045089721679687

In [26]:
print(embeddings[0])

[-0.00473785400390625, 0.025115966796875, -0.0267181396484375, 0.0278778076171875, 0.0234222412109375, -0.0138397216796875, 0.0361328125, -0.05145263671875, 0.02227783203125, -0.0295867919921875, 0.01062774658203125, 0.016937255859375, -0.061431884765625, -0.0122528076171875, 0.015777587890625, -0.01316070556640625, -0.05706787109375, 0.0018873214721679688, -0.011138916015625, -0.0259246826171875, 0.020538330078125, -0.053924560546875, 0.007251739501953125, 0.036407470703125, 0.016510009765625, 0.01418304443359375, 0.0706787109375, -0.023681640625, 0.0222320556640625, -0.02752685546875, 0.0419921875, -0.020538330078125, 0.045074462890625, -0.057159423828125, 0.038726806640625, 0.0204925537109375, 0.01605224609375, 0.047088623046875, -0.0250701904296875, -0.0135498046875, 0.0016918182373046875, 0.0262298583984375, 0.01031494140625, 0.019775390625, 0.031463623046875, 0.03851318359375, -0.046661376953125, 0.02899169921875, 0.0296630859375, 0.03070068359375, 0.01251220703125, 0.04891967773

### 코사인유사도 검사

In [27]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [28]:
query = ["Rainbow is beautiful"]

In [29]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.15572177657821906
0.20517060747601842
0.18451509213180314


In [30]:
query = ["무지개는 아름다워"]

In [31]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.25181293882298333
0.4561053681917829
0.1952532758819349


In [32]:
e_query = embedding_model.embed_documents("컴퓨터는 역시 사과 브랜드지")
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.1824960070116332
0.16273522335036136
0.20096169287309418


In [33]:
e_query = embedding_model.embed_documents("사과컴퓨터가 창문컴퓨터보다 좋아")
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.19105199407799253
0.15242395723667987
0.16411162238854682


In [34]:
e_query = embedding_model.embed_documents("빨간 바나나")
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.3467303066625774
0.1969136933013044
0.18150252836650133


In [35]:
e_query = embedding_model.embed_documents("파란 컴퓨터")
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.16828573212731007
0.19816497056045287
0.30526857916577743


In [37]:
from langchain_chroma import Chroma

In [36]:
# 파일 load 부분 위에서 실행함.
# loader = PyPDFLoader("./content/Demian.pdf")
# pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [38]:
db = Chroma.from_documents(docs, embedding_model)

### 쿼리 요청해보기

In [39]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [40]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

In [41]:
query = "who wrote the book?"
docs = db.similarity_search(query)

In [42]:
print(docs[0].page_content)

JACOB AND THE ANGEL 
body but it was not one which he himself could choose, 
re-cast and regulate to his own liking. One had no right 
to want new gods, no right at all to want to give the 
world anything of that sort I There was but one duty 
for a grown man; it was to seek the way to himself, to 
become resolute within, to grope his way forward wher­
ever that might lead him. The discovery shook me pro­
foundly; it was the fruit of this experience. I had often 
toyed with pictures of the future, dreamed of r6les which 
might be assigned to me---as a poet, maybe, or prophet 
or painter or kindred vocation. All that was futile. I was 
not there to write poetry, to preach or paint; neither I 
nor any other man was there for that purpose. They 
were only incidental things. There was only one true 
vocation for everybody-to find the way to himself. He 
might end as poet, lunatic, prophet or criminal-that 
was not his affair; ultimately it was of no account. His 
affair was to discover his

In [43]:
query = "in what year the book was published? and who is the publisher? and what is the genre of the book?"
docs = db.similarity_search(query)

In [44]:
print(docs[0].page_content)

DEMIAN 
what it was. I passed through a suburb where there were 
brothels that had a light here and there in the windows. 
Farther out still stood some new buildings and piles of 
bricks partially covered with grey coloured snow. As I 
wandered like a somna111bulist under the influence of this 
strange impulse I noticed the new building of my home 
town; it was there that my tormentor Kromer had taken 
me for our first financial settlement. A similar building 
lay before me in the grey night, yawned at me with its 
gaping door. It drew me in; I had no choice and I 
advanced, stumbling over heaps of sand and rubbish; 
the urge was irresistible; I had to enter. I staggered 
into the dreary room over planks and broken bricks; it 
smelt depressingly of damp cold and stones. There was a 
heap of sand, a light stain of grey-all the rest was dark. 
Then a voice cried out in terror, "For God's sake, 
Sinclair, where have you come from?" 
And a figure rose out of the darkness, a small, lean 
fe

### Step 4 Retrievers

In [45]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


In [47]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-5-nano",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

### RAG 없이 LLM에 물어보기

In [48]:
response = llm.invoke("Demian이 어떤 책이야?")
print(response.content)

간단히 말해, Demian은 Hermann Hesse가 쓴 성장소설이다. 원제는 Demian. Die Geschichte von Emil Sinclairs Jugend(1919년 발표).

줄거리 요약
- 주인공 Emil Sinclair가 어릴 적부터 자아를 찾아가는 과정을 따라간다.
- 신비롭고 도발적인 친구 Demian이 나타나 “빛의 세계”와 “어둠의 세계” 같은 이분법을 넘나들며 Sinclair의 내면 성장을 돕는다.
- 전통적 도덕관에 의문을 품고 자기만의 길을 찾는 여정을 주요 축으로 삼는다.
- Abraxas 같은 신화적·상징적 아이디어를 통해 선과 악의 경계가 모호해지는 통합의 문제를 탐구한다.

주요 주제와 특징
- 자아 발견과 자기실현: 규범에 얽매이지 않고 진정한 자아를 찾는 과정.
- 양극의 통합: 빛/어둠, 선/악의 대비를 넘어서는 이해를 모색.
- 종교와 철학의 탐구: 인간 내면의 신성성, 영적 성장을 다룸.
- 심리적·상징적 문체: 상징이 많고 은유적 서사가 두드러진다.
- Hesse의 초기 대표작으로, Steppenwolf나 Siddhartha 같은 later 작품들로 가는 길목이 됨.

읽는 사람에게
- 심리적, 철학적 주제에 관심이 많고 자아 탐구를 즐기는 독자에게 추천.
- 짧은 분량이지만 상징이 많아 해설이나 주석이 도움이 될 수 있다.

참고
- 원제: Demian. Die Geschichte von Emil Sinclairs Jugend
- 영어 제목: Demian: The Story of Emil Sinclair's Youth
- 1919년 발표작으로, 1차 세계대전 직후의 시대적 맥락도 반영한다.간단히 말해, Demian은 Hermann Hesse가 쓴 성장소설이다. 원제는 Demian. Die Geschichte von Emil Sinclairs Jugend(1919년 발표).

줄거리 요약
- 주인공 Emil Sinclair가 어릴 적부터 자아를 찾아가는 과정을 따라간다.
- 신비롭고 도발적인 친구 Demian이 

### RAG 적용

In [49]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

In [51]:
response = qa("Demian이 어떤 책이야?")

Demian은 헤르만 헤세가 쓴 소설로, 성장기 소년 에밀 싱클레어의 자기 발견과 내면 세계를 탐구하는 이야기야. 1919년에 발표되었고, 모더니즘 분위기의 심리적·철학적 성찰이 강하게 드러나지.

주요 내용 간단히
- 주인공: 에밀 싱클레어, 겉으로는 밝고 안전한 세계와 속으로는 어둡고 위험한 충동 사이에서 갈등하는 소년.
- 핵심 인물: 신비롭고 강렬한 동급생 맥스 데미안. 데미안은 전통적 도덕이나 종교의 경계에 의문을 던지게 하는 인물로 싱클레어의 사고를 크게 바꾼다.
- 주제: 자아의 통합과 자유로운 사고, 도덕적 절대성에 대한 의문, 두 세계(빛과 어둠, 선과 악) 사이의 긴장, 그리고 내면의 “다이몬(daimon)”이라는 길잡이가 삶을 이끈다는 아이디어.
- 상징과 영향: 데미안은 Nietzsche나 Novalis 같은 철학적·문학적 영향에서 영감을 받아, 외부의 규범을 벗어나 자기만의 길을 찾는 과정을 상징적으로 다룬다. 이야기 속에서 Beatrice, Eva 같은 인물·상징도 싱클레어의 내면 여정과 연관된다.

전체 분위기
- 이야기 톤은 사색적이고 시적이며, 자아 발견과 영적 성장을 중심으로 전개된다. 독자에게 자기 자신과 세계에 대한 의문을 던지는 작품이야.

원한다면:
- 줄거리의 챕터별 요약, 등장인물 관계도, 주요 상징에 대한 해설도 자세히 정리해 줄 수 있어. 어떤 부분이 궁금한지 말해줘.

In [52]:
response = qa("Demian이라는 책을 아직 안봤는데, 관심가질수있게 왜 봐야하는지 흥미를 돋구어봐")

좋아요. Demian은 아직 읽지 않았다면, 한 번쯤 방문해볼 만한 매력적인 작품이에요. 간단하게 왜 관심을 가져볼 만한지 흥미를 돋우는 포인트를 정리해볼게요.

- 자아 발견의 여정을 담은 이야기
  어른이 되어 가는 과정에서 마주치는 불안, 도덕적 의심, 그리고 “나의 진짜 나”를 찾으려는 마음을 조용하고도 강렬하게 그려냅니다. 부모님과 학교가 가르친 바와는 다른 방향으로 자신을 이해하고 받아들이는 과정이 중심이에요.

- 두려움과 욕망, 빛과 어둠의 이중성
  이야기 속에서 선과 악, 옳고 그름은 단순한 이분법이 아니라 복잡하게 얽혀 있습니다. 그러한 모호함을 통해 독자는 자기 내면의 두려움과 욕망을 직면하게 되죠. 독서 내내 은유와 상징이 등장해 머릿속에 이미지를 남깁니다.

- “daimon”이라는 흥미로운 Concept
  주인공이 자신을 이끄는 내면의 목소리나 운명을 의미하는 daimon 같은 개념에 천천히 다가가면서, 자기 자신과의 대화를 시작하게 됩니다. 이내 당신도 자신의 인생 길에서 어떤 ‘다이몬’을 만나는지 생각해보게 될 거예요.

- 간결한 문장 속의 깊은 철학
  니체와 노발리스 같은 사상가들의 영향이 읽는 재미를 더합니다. 철학적 질문들이 자연스럽게 문장 속에 녹아 있어, 생각을 곱씹는 즐거움이 큽니다. 아주 어렵지 않으면서도 생각의 깊이가 깊은 편이에요.

- 짧은 분량으로도 큰 여운
  비교적 짧은 분량의 소설이므로 부담 없이 시작하기 좋아요. 몰입감은 강력하고, 읽고 나면 오랫동안 떠올릴 만한 주제들을 남깁니다.

- 어떤 독자에게 특히 잘 맞을까
  - 자기 정체성이나 신념이 흔들리는 순간을 겪고 있는 사람
  - 전통적 가치관에 의심이 생겼을 때 새로운 관점이 필요했던 사람
  - 심리적, 철학적 주제에 매혹되는 독자
  - 문학적 상징과 은유를 즐기는 독자

읽기 시작하기 팁
- 처음부터 끝까지 풀 스토리보다는, 챕터별로 작은 질문을 던지며 읽어보면 더 흥미롭게 다가옵니다. 예를 들어 “왜 Demian은 그렇게 행동하는가?

In [53]:
response = llm.invoke("Demian이라는 책을 아직 안봤는데, 관심가질수있게 왜 봐야하는지 흥미를 돋구어봐")
print(response.content)

좋아요. Demian은 한 번 읽으면 오랫동안 남는, 자기 자신을 찾아가는 여정의 문학작품이에요. 짧은 분량이지만 생각할 거리와 감정의 깊이가 아주 깊습니다. 흥미를 돋구는 핵심 포인트들로 소개해볼게요.

- 간단한 줄거리 포인트
  싱클레어라는 남학생이 낡은 도덕과 사회적 기대 속에서 자아를 잃지 않으려 애쓰다가, 신비롭고 반항적인 친구 데미안과 만나면서 자신만의 길을 찾아가는 이야기예요. 겉으로 보기엔 평범한 성장소설 같지만, 마음의 어둠과 빛을 어떻게 통합할지, 무엇이 “옳다”는 것인지에 대해 깊이 탐구합니다.

왜 읽어야 하는지—흥미 포인트
- 자아 정체성의 대장정: 어릴 적 안심감을 벗어나 자신만의 진실을 추구하는 과정을 생생하게 따라가요. 나이 들어가며 누구나 한 번쯤 겪는 “내가 정말 원하는 건 무엇인가?”라는 물음에 직접 다가갑니다.
- 내면의 양극화와 통합의 아이디어: 선과 악, 빛과 어둠 같은 이분법을 넘어서는 생각, 즉 ‘두 가지를 하나로 합치는 자아의 통합’이라는 주제를 상징적으로 다뤄요. 이 부분이 읽는 내내 뇌리에서 맴돌 수 있습니다.
- 상징과 꿈의 세계를 맛볼 수 있는 문장들: 은유와 상징이 풍부하고, 독특한 분위기의 문체가 독자의 상상력을 자극합니다. 한 문장 한 문장이 마치 그림처럼 다가와 생각하게 만듭니다.
- 심리학적 사유의 씨앗: 자아의 독립성과 사회의 기대 사이에서 균형을 찾는 여정은 현대로 와도 충분히 공감 가능한 이야기예요. 읽고 나면 “나의 진짜 목소리는 어디에 있는가” 같은 질문이 머릿속에 남습니다.
- 역사적 맥락과 보편성의 만남: 1차 세계대전 직전·후의 불안과 변화 속에서 인간의 본성에 대한 심오한 성찰이 녹아 있습니다. 시대를 뛰어넘어 오늘날의 독자도 여전히 울리는 메시지를 담고 있어요.
- 짧고 몰입도 높은 구성: 분량이 비교적 짧아 부담 없이 시작하기 좋습니다. 한두 주말 사이에 끝내고도 여운과 생각거리를 오래 남길 수 있어요.
- 영향을 준 작품으로서의 가치: 이 책은 구체적으로 Jungian 심리학의

In [54]:
QUERY_TEXT = "Demian 책 내용을 10줄 요약해줘."

In [55]:
response = qa(QUERY_TEXT)

- 싱클레어는 프란츠 크로메르의 협박과 죄책감 속에서 도덕적 갈등을 겪는다.  
- 신비롭고 부유한 소년 데미안이 등장하며, 그의 주변은 소문과 미스터리로 가득해진다.  
- 데미안은 신과 악마의 이원성과 전통적 교리에 얽매이지 않는 삶의 가능성을 보여준다.  
- 크로메르의 위협은 싱클레어를 현실의 어둠으로 끌어들여 그의 세계관에 균열을 만든다.  
- 싱클레어는 두 세계, 즉 빛의 세계와 어둠의 세계의 존재를 받아들이기 시작한다.  
- 데미안과의 상호작용과 독서를 통해 자아의 내면에 잠든 운명적 자아(다이몬)를 인식한다.  
- 벽에 Beatrice(또는 Demian)의 초상을 붙여 보며, 그 얼굴이 결국 자신의 삶과 운명을 반영한다고 느낀다.  
- 니체와 노발리스 등의 영향을 받으며 삶의 문제를 더 깊이 성찰하게 된다.  
- 이 과정은 종교적 권위와 사회적 기대를 넘어서는 자립과 자기 책임의 의식을 키운다.  
- 결국 싱클레어는 평범한 소년에서 자신의 길을 스스로 선택하는 독립적 자아로 성장한다.

In [56]:
response = llm.invoke(QUERY_TEXT)
print(response.content)

주인공 소년은 전통적 도덕관과 내면의 갈등 속에서 자아를 찾아가는 이야기가 시작된다.
수수께끼 같은 동급생 데미안은 도덕의 절대성에 의문을 던지며 그의 내적 각성을 촉진한다.
피스토리우스라는 오르간 연주자의 도움으로 예술과 영성에 대한 탐구가 본격적으로 시작된다.
선과 악의 이원론을 넘어서는 여정에서 아브락사스라는 통합의 상징이 등장한다.
여성 인물 에바는 영적 이상향이자 통합의 상징으로 주인공의 길잡이가 된다.
주인공은 중산층의 규범을 넘어 자기다움을 스스로 선택하는 길을 걷기 시작한다.
음악과 문학, 신비를 통해 내면의 목소리에 귀를 기울이며 자아를 재구성한다.
성장은 고뇌를 수반하고 외부 권위가 아닌 자신의 결단이 진리의 기준이 된다는 깨달음이 커진다.
철학과 종교의 만남 속에서 세계관이 확장되며 자아의 자유가 중시된다.
마지막에 그는 성숙한 자아를 받아들이고 독립적 삶의 길을 선택하며, 데미안은 여운 있는 길잡이로 남는다.주인공 소년은 전통적 도덕관과 내면의 갈등 속에서 자아를 찾아가는 이야기가 시작된다.
수수께끼 같은 동급생 데미안은 도덕의 절대성에 의문을 던지며 그의 내적 각성을 촉진한다.
피스토리우스라는 오르간 연주자의 도움으로 예술과 영성에 대한 탐구가 본격적으로 시작된다.
선과 악의 이원론을 넘어서는 여정에서 아브락사스라는 통합의 상징이 등장한다.
여성 인물 에바는 영적 이상향이자 통합의 상징으로 주인공의 길잡이가 된다.
주인공은 중산층의 규범을 넘어 자기다움을 스스로 선택하는 길을 걷기 시작한다.
음악과 문학, 신비를 통해 내면의 목소리에 귀를 기울이며 자아를 재구성한다.
성장은 고뇌를 수반하고 외부 권위가 아닌 자신의 결단이 진리의 기준이 된다는 깨달음이 커진다.
철학과 종교의 만남 속에서 세계관이 확장되며 자아의 자유가 중시된다.
마지막에 그는 성숙한 자아를 받아들이고 독립적 삶의 길을 선택하며, 데미안은 여운 있는 길잡이로 남는다.


In [57]:
QUERY_TEXT = "아직 읽고싶은 마음이 안들어. 소설속에서 정말 흥미로운 부분을 발췌해서 소개해줘."
response = qa(QUERY_TEXT)

좋아요. 아래는 Demian에서 가장 흥미로운 순간 중 하나로 꼽히는 “생각읽기” 장면의 짧은 발췌입니다. 원문 영어 일부를 그대로 옮겼으며, 이 부분은 자아 탐구와 두려움의 힘에 대해 시사합니다.

I am fond of you or interested in you and I would like to discover what it's like inside you. To do this I have already taken the initial step. I frightened you. You are therefore in a nervous state. How has that come about? You don't need to be afraid of anybody. When you are afraid of someone it means you have provided that someone with some kind of lever. You have done something wrong for example and the other person knows this and by this means has acquired a hold over you. You understand? It's clear enough, isn't it?

I told you thought-reading seems comic but it comes about quite naturally. I could for example tell you pretty accurately what you thought about me when I once told you the story of Cain and Abel. But that has nothing to do with the present case. I also consider it possible that you once dreamed about me. But let's for-

이 부분은 대화의 톤과 관계의 힘에 대한 예고를 주며, 나중에 드

In [58]:
response = llm.invoke(QUERY_TEXT)
print(response.content)

죄송하지만 요청하신 소설의 특정 구절을 그대로 발췌해 드리는 건 도와드리기 어렵습니다. 다만, 흥미를 끌 수 있는 핵심 내용이나 분위기를 느낄 수 있는 요약으로 소개해 드릴 수 있어요.

다음 중 하나로 도와드릴까요?
- 1) 제목이나 작가를 말해주시면, 그 부분의 핵심 이벤트와 캐릭터의 변화, 왜 읽고 싶은 마음이 생길지에 대한 요약으로 소개해 드립니다.
- 2) 분위기나 분위기(스릴러, 판타지, 로맨스 등)와 원하시는 톤을 알려주시면, 스포일러를 최소화한 짧은 프리뷰 읽기 소개를 만들어 드립니다.
- 3) 공공도메인 작품이나 특정 작품의 분위기와 주제를 중심으로 한 해설형 소개를 원하시면 그에 맞춰 정리해 드립니다.

먼저 몇 가지를 알려주실 수 있을까요?
- 읽고 싶은 소설의 제목이나 작가
- 선호하는 장르나 분위기
- 스포일러 여부에 대한 선호(스포일러 없이 시작하고 싶다 등)

원하시는 방향을 알려주시면 바로 그에 맞춘 흥미 포인트 중심의 소개를 만들어 드릴게요.죄송하지만 요청하신 소설의 특정 구절을 그대로 발췌해 드리는 건 도와드리기 어렵습니다. 다만, 흥미를 끌 수 있는 핵심 내용이나 분위기를 느낄 수 있는 요약으로 소개해 드릴 수 있어요.

다음 중 하나로 도와드릴까요?
- 1) 제목이나 작가를 말해주시면, 그 부분의 핵심 이벤트와 캐릭터의 변화, 왜 읽고 싶은 마음이 생길지에 대한 요약으로 소개해 드립니다.
- 2) 분위기나 분위기(스릴러, 판타지, 로맨스 등)와 원하시는 톤을 알려주시면, 스포일러를 최소화한 짧은 프리뷰 읽기 소개를 만들어 드립니다.
- 3) 공공도메인 작품이나 특정 작품의 분위기와 주제를 중심으로 한 해설형 소개를 원하시면 그에 맞춰 정리해 드립니다.

먼저 몇 가지를 알려주실 수 있을까요?
- 읽고 싶은 소설의 제목이나 작가
- 선호하는 장르나 분위기
- 스포일러 여부에 대한 선호(스포일러 없이 시작하고 싶다 등)

원하시는 방향을 알려주시면 바로 그에 맞춘 흥미 포인트 중심의 소개를 만들어 드릴게요.


### 고성능 모드


In [62]:
llm_pro = ChatOpenAI(
    model="gpt-5.5-pro",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=False,              # 실시간 출력을 켭니다
)

In [63]:
response = llm_pro.invoke(QUERY_TEXT)
print(response.content)

[{'id': 'rs_0b7d85ef29a91fb6006a15584d0278819981c929f04936455b', 'summary': [], 'type': 'reasoning'}, {'type': 'text', 'text': '좋아. 그런데 지금 대화에 **어떤 소설인지 제목이 안 보여서** 바로 발췌를 고르긴 어려워.\n\n소설 **제목/작가**만 알려줘. 그러면 내가:\n\n1. **가장 흥미로운 장면을 짧게 발췌**해서 보여주고  \n2. 그 장면이 왜 매력적인지 설명하고  \n3. **스포일러 없이 / 약스포 / 강스포** 중 원하는 방식으로 소개해줄게.\n\n다만 저작권 있는 현대 소설이면 원문을 길게 그대로 가져오긴 어렵고, 대신 **짧은 인용 + 생생한 장면 설명**으로 읽고 싶어지게 소개해줄 수 있어.  \n네가 원문 일부를 직접 보내주면 그 안에서는 더 길게 골라서 분석해줄 수도 있어.\n\n제목만 보내줘. 내가 “이 부분 읽으면 마음 바뀔 만하다” 싶은 장면으로 골라줄게.', 'annotations': [], 'id': 'msg_0b7d85ef29a91fb6006a15584d047081998cc5a428d9d72323', 'phase': 'final_answer'}]


In [66]:
from IPython.display import display, Markdown

def extract_final_text(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        texts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                texts.append(item.get("text", ""))
        return "\n".join(texts)

    return str(content)


response = llm_pro.invoke(QUERY_TEXT)

final_text = extract_final_text(response.content)

display(Markdown(final_text))

어떤 소설인지 제목/작가를 알려줘.  
지금 대화창에는 작품명이 안 보여서 정확히 발췌를 못 하겠어.

알려주면 내가 이렇게 해줄게:

- **스포일러 없이** 가장 흥미로운 장면 소개  
- **정말 읽고 싶게 만드는 짧은 원문 발췌**  
- 왜 그 장면이 매력적인지 간단히 설명  
- 원하면 **“이 소설이 나랑 맞을지”**도 판단해줄게

※ 저작권이 있는 현대 소설이면 긴 원문은 어렵지만, 아주 짧은 인상적인 문장과 장면 요약으로 대신 소개해줄 수 있어. 원문 일부를 네가 보내주면 그 안에서는 더 길게 골라줄 수 있고.

### Step 5 Question Answering

### Q
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### A
- 책의 내용을 인용/발췌 해달라고 했을때, RAG를 적용하면 실제 원문을 인용해서 답변을 해주었음.
- 솔직히 책내용을 몰라서 답변만 가지고 크게 판단하기 어려웠음.
### 기타 느낀점
- LLM에게 그냥 물어봐도 요약은 잘해주지만 인용은 해줄수없다고 말함.
- 모델을 gpt-4o, gpt-5-nano, gpt-5.5-pro 3가지 사용해봤는데, 이번 예제 용도로 사용하는데는 가장 저렴한 5-nano도 사용하는데 큰 문제가 없었음.